# FitzHugh-Nagumo: an excitable membrane

A two-variable caricature of a neuron: a fast voltage-like variable `v` with a cubic
nullcline and a slow recovery variable `w`. For small input `i` there is one stable rest
state and a large excursion (a spike) only after a big enough kick; as `i` grows the rest
state loses stability in a Hopf bifurcation and the cell fires repeatedly.

*Based on an example from XPP Studio (Peter Ashwin and Bard Ermentrout, GPL-3.0). Every number below is computed by XPPAUT 8.1 and its AUTO through pyauto-xpp.*

In [1]:
import plotly.graph_objects as go
from pyxpp.builder import ModelBuilder
from xpptut import Explorer, load, orbit_diagram, show_figure

## The model

Written with `ModelBuilder`, statement by statement as in an `.ode` file. Edit it and rerun the cells below.

In [2]:
mb = (ModelBuilder("fitzhugh_nagumo")
      .comment("FitzHugh-Nagumo excitable membrane")
      .ode("v", "v-v^3/3-w+i")
      .ode("w", "eps*(v+a-b*w)")
      .parameter(i=0.2, a=0.7, b=0.8, eps=0.08)
      .initial(v=-1.2, w=-0.6)
      .option(total=200, dt=0.05, xp="v", yp="w", xlo=-2.6, xhi=2.6, ylo=-1, yhi=2))
print(mb.render())

# FitzHugh-Nagumo excitable membrane
v'=v-v^3/3-w+i
w'=eps*(v+a-b*w)
par i=0.2,a=0.7,b=0.8,eps=0.08
init v=-1.2,w=-0.6
@ total=200,dt=0.05,xp=v,yp=w,xlo=-2.6,xhi=2.6,ylo=-1,yhi=2
done



## Explore

The sliders, the direction-field menu and the AUTO panel all call XPP. `ex.m` is the pyxpp model, for anything else.

In [3]:
ex = Explorer(mb, kind="plane", x="v", y="w", window=(-2.6, 2.6, -1, 2),
              sliders={"i": (0, 1.6), "eps": (0.005, 0.3), "a": (0, 1.5)},
              auto="i", auto_range=(0, 2), auto2="eps", auto2_range=(0, 0.3),
              title="FitzHugh-Nagumo")
ex

(run this cell with the kernel to get the interactive explorer)

## Going further

**Excitability from code.** Start at rest and kick `v` by increasing amounts: below a
threshold the cell returns quietly, above it it makes a full spike.

In [4]:
import numpy as np
rest = ex.m.equilibrium({"v": -1.2, "w": -0.6}).x
peaks = []
for kick in np.linspace(0.1, 1.2, 12):
    tr = ex.m.run(keep=False, total=60, v=rest["V"] + kick, w=rest["W"])
    peaks.append(tr["v"].max())
show_figure(go.Figure(go.Scatter(x=np.linspace(0.1, 1.2, 12), y=peaks, mode="lines+markers")),
            xaxis_title="kick in v", yaxis_title="largest v reached", height=300)

**Where repetitive firing starts.** AUTO finds the two Hopf points in `i`; the first
Lyapunov coefficient says whether the orbits are born stable (supercritical) or unstable.

In [5]:
ex.set(i=0.2)
parts = ex.run_auto()
for p in [p for d in parts for p in d.special("HB")]:
    l1, om = ex.m.auto.lyapunov(p)
    print(f"Hopf at i = {p.x:.5f}: l1 = {l1:+.4f} ({'sub' if l1 > 0 else 'super'}critical), period at onset {2*np.pi/om:.3f}")

Hopf at i = 0.33128: l1 = +0.9719 (subcritical), period at onset 22.806


Hopf at i = 1.41872: l1 = +0.9720 (subcritical), period at onset 22.806


## Things to try

- Move `i` past about 0.33 and watch the rest state turn into a spiral source.
- Choose **flow (streamlines)** or **speed map + flow** to see the slow and fast parts of the cycle.
- Make `eps` small (0.01): the cycle hugs the cubic nullcline (relaxation oscillations).
- In the AUTO panel, continue equilibria, click an HB point, and get its periodic orbits.